# Aurisign FIlipino Sign Language Gesture Recognition 1D CNN Alphabets Training

## Dependency Installation and Imports

In [2]:
!pip install opencv-python mediapipe==0.10.14 matplotlib seaborn tensorflow scikit-learn tensorflowjs


[notice] A new release of pip is available: 26.0 -> 26.2
[notice] To update, run: pip install --upgrade pip


In [3]:
# %%
import cv2
import mediapipe as mp
import numpy as np
import os
from time import time

2026-07-30 21:58:14.411433: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-30 21:58:14.525575: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-30 21:58:14.663231: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1785419894.809951   30691 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1785419894.852829   30691 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1785419895.106684   30691 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

### Data Collection

In [6]:
# %%
# Motion gesture data collection (J, Z) — captures short SEQUENCES instead of single frames.
# Press "c" to record one full sequence for the current letter, "q" to move to the next letter.
MOTION_CLASSES = ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J", "K", "L", "M", "N", "Ñ", "Ng", "O", "P", "Q", "R", "S", "T", "U", "V", "W", "X", "Y", "Z"]   # full set — used everywhere downstream

MOTION_DATA_PATH = "alphabet_dataset"
SEQ_LEN = 24                 # ~0.8-1s of motion at a typical webcam framerate
SAMPLES_PER_GESTURE = 100    # sequences, not frames
CENTER_BY_WRIST = True
INCLUDE_HANDEDNESS = True
NUM_FEATURES = 64 if INCLUDE_HANDEDNESS else 63

os.makedirs(MOTION_DATA_PATH, exist_ok=True)
for g in MOTION_CLASSES:
    os.makedirs(os.path.join(MOTION_DATA_PATH, g), exist_ok=True)

mp_hands = mp.solutions.hands
mp_draw  = mp.solutions.drawing_utils
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1,
                       min_detection_confidence=0.7, min_tracking_confidence=0.7)

cap = cv2.VideoCapture(0)

def get_frame_keypoints(frame):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)
    if not result.multi_hand_landmarks:
        return None, frame
    lm = result.multi_hand_landmarks[0]
    mp_draw.draw_landmarks(frame, lm, mp_hands.HAND_CONNECTIONS)
    pts = np.array([[p.x, p.y, p.z] for p in lm.landmark])
    if CENTER_BY_WRIST:
        pts -= pts[0]
    pts = pts.flatten()
    if INCLUDE_HANDEDNESS and result.multi_handedness:
        hl = result.multi_handedness[0].classification[0].label
        pts = np.append(pts, 0 if hl == "Left" else 1)
    return pts, frame

for gesture in MOTION_CLASSES:
    print(f"Collecting motion: {gesture} — [c]=record one {SEQ_LEN}-frame sequence, [q]=next letter")
    count = 0
    while count < SAMPLES_PER_GESTURE:
        ret, frame = cap.read()
        if not ret:
            continue
        frame = cv2.flip(frame, 1)
        _, frame = get_frame_keypoints(frame.copy())
        cv2.putText(frame, f"{gesture} {count}/{SAMPLES_PER_GESTURE}  [c]=record  [q]=next",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        cv2.imshow("Collect Motion Gestures", frame)
        k = cv2.waitKey(1) & 0xFF

        if k == ord("c"):
            sequence = []
            print(f"  Recording {gesture} #{count}... perform the gesture now")
            while len(sequence) < SEQ_LEN:
                ret, f2 = cap.read()
                if not ret:
                    continue
                f2 = cv2.flip(f2, 1)
                kp, f2 = get_frame_keypoints(f2)
                # a short mid-gesture occlusion shouldn't corrupt the whole window,
                # so hold the last known keypoints instead of dropping the frame
                if kp is None and sequence:
                    kp = sequence[-1]
                elif kp is None:
                    kp = np.zeros(NUM_FEATURES)
                sequence.append(kp)
                cv2.putText(f2, f"RECORDING {len(sequence)}/{SEQ_LEN}", (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
                cv2.imshow("Collect Motion Gestures", f2)
                cv2.waitKey(1)
            np.save(os.path.join(MOTION_DATA_PATH, gesture, f"{count:03d}.npy"), np.array(sequence))
            count += 1

        if k == ord("q"):
            break

cap.release()
cv2.destroyAllWindows()
print("Done collecting motion gestures!")

I0000 00:00:1785422616.409022   30691 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1785422616.411264   95757 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.0.7-0ubuntu0.24.04.2), renderer: Mesa Intel(R) UHD Graphics 620 (WHL GT2)
W0000 00:00:1785422616.421966   95750 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1785422616.445073   95750 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


  Recording O #0... perform the gesture now
  Recording O #1... perform the gesture now
  Recording O #2... perform the gesture now
  Recording O #3... perform the gesture now
  Recording O #4... perform the gesture now
  Recording O #5... perform the gesture now
  Recording O #6... perform the gesture now
  Recording O #7... perform the gesture now
  Recording O #8... perform the gesture now
  Recording O #9... perform the gesture now
  Recording O #10... perform the gesture now
  Recording O #11... perform the gesture now
  Recording O #12... perform the gesture now
  Recording O #13... perform the gesture now
  Recording O #14... perform the gesture now
  Recording O #15... perform the gesture now
  Recording O #16... perform the gesture now
  Recording O #17... perform the gesture now
  Recording O #18... perform the gesture now
  Recording O #19... perform the gesture now
  Recording O #20... perform the gesture now
  Recording O #21... perform the gesture now
  Recording O #22...

### Alphabet Data Loading and Preprocessing

In [17]:
# %%
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

X_motion, y_motion = [], []
for i, gesture in enumerate(MOTION_CLASSES):
    folder = os.path.join(MOTION_DATA_PATH, gesture)
    for f in os.listdir(folder):
        X_motion.append(np.load(os.path.join(folder, f)))
        y_motion.append(i)

X_motion = np.array(X_motion)   # shape: (num_samples, SEQ_LEN, NUM_FEATURES)
y_motion = to_categorical(y_motion, len(MOTION_CLASSES))

Xm_train, Xm_test, ym_train, ym_test = train_test_split(
    X_motion, y_motion, test_size=0.2, random_state=42)
print("Train:", Xm_train.shape, "Test:", Xm_test.shape)

Train: (2240, 24, 64) Test: (560, 24, 64)


### Alphabet Model

In [18]:
# %%
motion_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(SEQ_LEN, X_motion.shape[-1])),
    tf.keras.layers.Conv1D(64, 3, activation='relu', padding='same'),
    tf.keras.layers.Conv1D(64, 3, activation='relu', padding='same'),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(MOTION_CLASSES), activation='softmax'),
])
motion_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
motion_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_4 (Conv1D)               │ (None, 24, 64)         │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_5 (Conv1D)               │ (None, 24, 64)         │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_2      │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 28)             │         1,820 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,684 (119.86 KB)

 Trainable params: 30,684 (119.86 KB)

 Non-trainable params: 0 (0.00 B)

### Alphabet Model Training

In [19]:
# %%
motion_history = motion_model.fit(Xm_train, ym_train, validation_data=(Xm_test, ym_test),
                                   epochs=50, batch_size=8)

Epoch 1/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.2286 - loss: 2.6338 - val_accuracy: 0.5429 - val_loss: 1.5228
Epoch 2/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5902 - loss: 1.2074 - val_accuracy: 0.8268 - val_loss: 0.6559
Epoch 3/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7388 - loss: 0.7340 - val_accuracy: 0.8768 - val_loss: 0.4000
Epoch 4/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8089 - loss: 0.5278 - val_accuracy: 0.8714 - val_loss: 0.3503
Epoch 5/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8647 - loss: 0.3858 - val_accuracy: 0.9304 - val_loss: 0.2269
Epoch 6/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8781 - loss: 0.3371 - val_accuracy: 0.9304 - val_loss: 0.1937
Epoch 7/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9040 - loss: 0.2548 - val_accuracy: 0.9321 - val_loss: 0.1905
Epoch 8/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9071 - loss: 0.2377 - val_accuracy: 0.

### Alphabet Model Export

In [21]:
# %%
motion_model.export("alphabet_model")
print("Alphabet SavedModel export complete!")

# In Google Colab, convert this the same way you convert the static model,
# just pointed at the motion folders so it doesn't overwrite the static tfjs export:
# !tensorflowjs_converter --input_format=tf_saved_model --output_format=tfjs_graph_model alphabet_model tfjs_alphabet_model

INFO:tensorflow:Assets written to: alphabet_model/assets


INFO:tensorflow:Assets written to: alphabet_model/assets


Saved artifact at 'alphabet_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 24, 64), dtype=tf.float32, name='keras_tensor_14')
Output Type:
  TensorSpec(shape=(None, 28), dtype=tf.float32, name=None)
Captures:
  129259997200656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129259997204112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129259997205072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129259997204304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129259997205456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129259997205264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129259997205840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129259997205648: TensorSpec(shape=(), dtype=tf.resource, name=None)
Alphabet SavedModel export complete!
